# Applying logistic regression on the final dataset

In [5]:
import pandas as pd
import numpy as np

df=pd.read_csv("../data/processed/reduced/Final_data.csv")

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
from sklearn.calibration import calibration_curve

num = ['target_runs', 'runs_left', 'CRR', 'balls_left', 'RRR',
       'wickets_left', 'runs_last_30', 'wickets_last_30']

X, y = df[num], df['won']

# split by match so no match appears in both train and test
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                           .split(X, y, groups=df['match_id']))

model = Pipeline([('scale', StandardScaler()),
                  ('clf', LogisticRegression(max_iter=2000))])
model.fit(X.iloc[train_idx], y.iloc[train_idx])

p = model.predict_proba(X.iloc[test_idx])[:, 1]      # P(chasing team wins)
y_test = y.iloc[test_idx]

print('accuracy', accuracy_score(y_test, p > 0.5))
print('roc-auc ', roc_auc_score(y_test, p))
print('log loss', log_loss(y_test, p))
print('brier   ', brier_score_loss(y_test, p))

accuracy 0.801850027836067
roc-auc  0.8971884524630729
log loss 0.403488973874876
brier    0.13165449949968566


In [3]:
# do predicted probabilities match reality? (each pair should be close)
frac_won, mean_pred = calibration_curve(y_test, p, n_bins=10)
print(list(zip(mean_pred.round(2), frac_won.round(2))))

# more reliable than one split
cv_p = cross_val_predict(model, X, y, groups=df['match_id'],
                         cv=GroupKFold(5), method='predict_proba')[:, 1]
print(roc_auc_score(y, cv_p), log_loss(y, cv_p))

[(np.float64(0.03), np.float64(0.02)), (np.float64(0.15), np.float64(0.17)), (np.float64(0.25), np.float64(0.25)), (np.float64(0.35), np.float64(0.33)), (np.float64(0.45), np.float64(0.43)), (np.float64(0.55), np.float64(0.51)), (np.float64(0.65), np.float64(0.64)), (np.float64(0.75), np.float64(0.76)), (np.float64(0.85), np.float64(0.92)), (np.float64(0.96), np.float64(1.0))]
0.8751404290569467 0.4368047966680802


## Experiment: do the last-30-balls features matter?
Compare the full 8-feature model with versions that drop them, using
match-grouped 5-fold CV (no match appears in both train and test).

In [6]:
ALL = ['target_runs', 'runs_left', 'CRR', 'balls_left', 'RRR',
       'wickets_left', 'runs_last_30', 'wickets_last_30']
y, groups = df['won'], df['match_id']

def cv_scores(cols):
    rows = []
    for tr, te in GroupKFold(n_splits=5).split(df, y, groups):
        pipe = Pipeline([('scaler', StandardScaler()),
                         ('model', LogisticRegression(max_iter=2000))])
        pipe.fit(df[cols].iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(df[cols].iloc[te])[:, 1]
        rows.append((log_loss(y.iloc[te], p),
                     brier_score_loss(y.iloc[te], p),
                     accuracy_score(y.iloc[te], p > 0.5)))
    return np.array(rows)

base = cv_scores(ALL)
print(f"{'all 8':14s}", base.mean(axis=0).round(4))

experiments = {
    'drop both':   ['runs_last_30', 'wickets_last_30'],
    'drop runs30': ['runs_last_30'],
    'drop wkts30': ['wickets_last_30'],
}
for name, drop in experiments.items():
    r = cv_scores([c for c in ALL if c not in drop])
    worse = int((r[:, 0] > base[:, 0]).sum())
    print(f"{name:14s}", r.mean(axis=0).round(4), f"worse in {worse}/5 folds")

all 8          [0.4368 0.1442 0.7841]
drop both      [0.4373 0.1443 0.7853] worse in 4/5 folds
drop runs30    [0.4369 0.1441 0.7849] worse in 4/5 folds
drop wkts30    [0.4372 0.1443 0.7845] worse in 3/5 folds


**Conclusion:** dropping both features changes log loss by about 0.0005, which is
negligible, so the deployed model uses the 6 match-state features only. Which is simpler and much safer for the user.